# Parkinson's Disease Dataset — Exploratory Data Analysis (EDA)

This notebook conducts a comprehensive data inspection, cleaning, quality check, and exploratory visual analysis on the Parkinson's Disease clinical dataset.

## 1. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
%matplotlib inline

## 2. Load Dataset

Dataset location relative to notebook: `../DATASET/parkinsons_dataset.csv`.

In [ ]:
df = pd.read_csv("../DATASET/parkinsons_dataset.csv")
print("Dataset shape:", df.shape)

## 3. Dataset Understanding

Inspect first 5 rows, last 5 rows, data types, and column names.

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
print("Column Names:\n", df.columns.tolist())
print("\nData Types:\n", df.dtypes)

In [ ]:
df.info()

In [ ]:
df.describe().T

## 4. Data Cleaning & Integrity Check

Check for missing values, duplicates, and invalid columns.

In [ ]:
print("Missing values per column:\n", df.isnull().sum())
print("\nTotal missing values:", df.isnull().sum().sum())

In [ ]:
print("Duplicate rows count:", df.duplicated().sum())

In [ ]:
# Drop trailing unnamed columns if present
unnamed = [c for c in df.columns if str(c).startswith("Unnamed")]
if unnamed:
    df = df.drop(columns=unnamed)

# Drop exact duplicates
df = df.drop_duplicates()
print("Shape after cleaning:", df.shape)

## 5. Identifier Column Handling

`PatientID` is a unique record identifier used for referencing, but **must not be used as a predictive feature** in machine learning modeling.

In [ ]:
if "PatientID" in df.columns:
    print("PatientID column detected. Unique IDs:", df["PatientID"].nunique())
    print("PatientID will be excluded during X/y feature separation.")

## 6. Exploratory Visualizations

### Target Distribution (Diagnosis)

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x="Diagnosis", palette="Set2", hue="Diagnosis", legend=False)
plt.title("Target Distribution: Parkinson's Diagnosis (0 = No, 1 = Yes)")
plt.xlabel("Diagnosis")
plt.ylabel("Patient Count")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom')
plt.show()

print("Diagnosis value counts:")
print(df["Diagnosis"].value_counts(normalize=True))

### Demographics Distribution (Age, Gender, BMI)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df["Age"], kde=True, ax=axes[0], color="skyblue")
axes[0].set_title("Age Distribution")

sns.countplot(data=df, x="Gender", ax=axes[1], palette="pastel", hue="Gender", legend=False)
axes[1].set_title("Gender Distribution (0=Female, 1=Male)")

sns.histplot(df["BMI"], kde=True, ax=axes[2], color="teal")
axes[2].set_title("BMI Distribution")
plt.tight_layout()
plt.show()

### Correlation Matrix Heatmap

In [ ]:
plt.figure(figsize=(14, 10))
corr = df.drop(columns=["PatientID"], errors="ignore").corr()
sns.heatmap(corr, cmap="coolwarm", annot=False, linewidths=0.5)
plt.title("Correlation Matrix of Clinical Features")
plt.show()

### Key Neurological Scores vs Diagnosis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=df, x="Diagnosis", y="UPDRS", ax=axes[0], palette="Set1", hue="Diagnosis", legend=False)
axes[0].set_title("UPDRS Score by Diagnosis")

sns.boxplot(data=df, x="Diagnosis", y="MoCA", ax=axes[1], palette="Set2", hue="Diagnosis", legend=False)
axes[1].set_title("MoCA Score by Diagnosis")
plt.tight_layout()
plt.show()

## 7. Machine Learning Preparation

Separate features `$X$` (32 clinical predictors) and target `$y$` (`Diagnosis`).

In [ ]:
X = df.drop(columns=["Diagnosis", "PatientID"], errors="ignore")
y = df["Diagnosis"]

print("Features X shape:", X.shape)
print("Target y shape:", y.shape)
print("Feature Columns:\n", X.columns.tolist())